<a href="https://colab.research.google.com/github/lmansf/Churn-Modeling/blob/main/Churn_on_Sample_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [106]:
import pandas as pd # dataframe manipulation
import numpy as np # number manipulation
import matplotlib.pyplot as plt # basic plots
import seaborn as sns #advanced plots

from sklearn.model_selection import train_test_split # for splitting dataset into training and validation
from sklearn.metrics import confusion_matrix, accuracy_score # How we're measuring performance of our models
from sklearn.model_selection import cross_val_score # For measuring performance of our cross validation
from sklearn.utils import resample #resampling library for handling our unbalanced dataset

from xgboost import XGBClassifier #XGBoost model
from sklearn.svm import SVC #SVC model
from sklearn.model_selection import GridSearchCV # Gridsearch is a tool that searches for the best hyperparameters of models. We'll use it for SVC tuning

# Clean Datasets
Goals:
- Import all necessary datasets
- Remove unused columns
- Filter out NA values
- Filter for passholders who we have records of renewing/not renewing
- Create custom columns for Recency, Frequency, Days to Expiration
- Merge tables
- Split dataset based on a target date. Everything before will be training/validation data, everything after will be validation data
- Undersample for renewed examples (balance the dataset to represent renewals/nonrenewals equally)
- Split the training data using K-folds

## Import Datasets

In [107]:
raw = pd.read_csv('/content/telecom_churn.csv')

In [108]:
df_raw = pd.DataFrame(raw)

In [109]:
# Making them Pandas DataFrames for manipulation
new_order = ['AccountWeeks','ContractRenewal','DataPlan','DataUsage','CustServCalls','DayMins','DayCalls','MonthlyCharge','OverageFee','RoamMins','Churn']
df_master = df_raw[new_order]

In [136]:
df_master

,AccountWeeks,ContractRenewal,DataPlan,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins,Churn
0,128,1,1,2.70,1,265.1,110,89.0,9.87,10.0,0
1,107,1,1,3.70,1,161.6,123,82.0,9.78,13.7,0
2,137,1,0,0.00,0,243.4,114,52.0,6.06,12.2,0
3,84,0,0,0.00,2,299.4,71,57.0,3.10,6.6,0
4,75,0,0,0.00,3,166.7,113,41.0,7.42,10.1,0
...,...,...,...,...,...,...,...,...,...,...,...
3328,192,1,1,2.67,2,156.2,77,71.7,10.78,9.9,0
3329,68,1,0,0.34,3,231.1,57,56.4,7.67,9.6,0
3330,28,1,0,0.00,2,180.8,109,56.0,14.44,14.1,0
3331,184,0,0,0.00,2,213.8,105,50.0,7.98,5.0,0


In [110]:
df_master_training = df_master[(df_master['AccountWeeks'] >= 97)]
df_master_validation = df_master[(df_master['AccountWeeks'] < 97)]

## Filter & Drop unused Columns
- We drop any columns we don't need. Smaller dataset to process, and removes columns that don't provide value to the analysis
- We're also going to do a handful of manipulations to look at just the data we're interested in

## Balance Handling: Resampling

In [112]:
# Assume 'df' is a DataFrame and 'target' is a column with imbalanced classes
# Separate majority and minority classes
df_majority = df_master_training[df_master_training.Churn == 0]
df_minority = df_master_training[df_master_training.Churn == 1]

# Undersample majority class
df_majority_undersampled = resample(df_majority,
                                    replace=False,    # sample without replacement
                                    n_samples=len(df_minority), # match minority number
                                    random_state=123) # reproducible results

# Combine minority class and undersampled majority class
df_balanced = pd.concat([df_majority_undersampled, df_minority])

In [113]:
dataset = df_balanced
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

In [114]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

## K-Fold Validation

# Kernel SVM
Kernel SVM is a Support Vector Machine algorithm, which means that it uses a kernel function, mapping data on different hyperplanes to group clusters.

In [115]:
svm_classifier = XGBClassifier()
svm_classifier.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [116]:
svm_tuned_classifier = SVC(C= 0.75, kernel= 'linear', random_state = 0)
svm_tuned_classifier.fit(X_train, y_train)

SVC(C=0.75, kernel='linear', random_state=0)

In [117]:
y_pred = svm_tuned_classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[43  9]
 [12 46]]


0.8090909090909091

# XGBoost

In [118]:
xgb_classifier = XGBClassifier()
xgb_classifier.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [119]:
y_pred = xgb_classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[43  9]
 [11 47]]


0.8181818181818182

In [120]:
accuracies = cross_val_score(estimator = xgb_classifier, X = X_train, y = y_train, cv = 10)
print("Accuracy: {:.2f} %".format(accuracies.mean()*100))
print("Standard Deviation: {:.2f} %".format(accuracies.std()*100))

Accuracy: 83.34 %
Standard Deviation: 6.62 %


# 3rd Model TBD

In [121]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [122]:
from sklearn.naive_bayes import GaussianNB
nb_classifier = GaussianNB()
nb_classifier.fit(X_train, y_train)

GaussianNB()

In [123]:
accuracies = cross_val_score(estimator = nb_classifier, X = X_train, y = y_train, cv = 10)
print("Accuracy: {:.2f} %".format(accuracies.mean()*100))
print("Standard Deviation: {:.2f} %".format(accuracies.std()*100))

Accuracy: 77.19 %
Standard Deviation: 5.86 %


# 4th Model Option: Random Forest Regression

In [124]:
from sklearn.ensemble import RandomForestRegressor
regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
regressor.fit(X, y)

RandomForestRegressor(n_estimators=10, random_state=0)

# Ensemble Learning
Ensemble Learning is a machine learning method where multiple models "vote" on a result. For example, 3 models voting on a cat or dog must all say cat, or dog, for the votes to go through. The third option is 'uncertain' which happens if the models aren't unanimous. In this case, the records are sent over to a human in the loop to verify.
- In our case, we could default to uncertain cases going to the mail group by default, increasing our net cast.

## Consolidate predictions into new columns
- xg_prediction, svm_prediciton, model3_prediction

In [125]:
df_master

,AccountWeeks,ContractRenewal,DataPlan,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins,Churn
0,128,1,1,2.70,1,265.1,110,89.0,9.87,10.0,0
1,107,1,1,3.70,1,161.6,123,82.0,9.78,13.7,0
2,137,1,0,0.00,0,243.4,114,52.0,6.06,12.2,0
3,84,0,0,0.00,2,299.4,71,57.0,3.10,6.6,0
4,75,0,0,0.00,3,166.7,113,41.0,7.42,10.1,0
...,...,...,...,...,...,...,...,...,...,...,...
3328,192,1,1,2.67,2,156.2,77,71.7,10.78,9.9,0
3329,68,1,0,0.34,3,231.1,57,56.4,7.67,9.6,0
3330,28,1,0,0.00,2,180.8,109,56.0,14.44,14.1,0
3331,184,0,0,0.00,2,213.8,105,50.0,7.98,5.0,0


In [126]:
# model 1 predictions
df_master_validation['model1'] = xgb_classifier.predict(df_master_validation.iloc[:, :-1].values)

# model 2 predictions
df_master_validation['model2'] = svm_tuned_classifier.predict(df_master_validation.iloc[:, :-2].values)

# model 3 predictions
df_master_validation['model3'] = regressor.predict(df_master_validation.iloc[:, :-3].values)

df_master_validation.head()

/tmp/ipython-input-2493646066.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_master_validation['model1'] = xgb_classifier.predict(df_master_validation.iloc[:, :-1].values)
/tmp/ipython-input-2493646066.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_master_validation['model2'] = svm_tuned_classifier.predict(df_master_validation.iloc[:, :-2].values)
/tmp/ipython-input-2493646066.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row

,AccountWeeks,ContractRenewal,DataPlan,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins,Churn,model1,model2,model3
3,84,0,0,0.00,2,299.4,71,57.0,3.10,6.6,0,1,1,0.6
4,75,0,0,0.00,3,166.7,113,41.0,7.42,10.1,0,0,1,0.7
10,65,1,0,0.29,4,129.1,137,44.9,11.43,12.7,1,1,1,1.0
11,74,1,0,0.34,0,187.7,127,49.4,8.17,9.1,0,0,0,0.5
13,95,1,0,0.44,3,156.6,88,52.4,12.38,12.3,0,1,1,0.4


## Define a column with Ensemble logic


In [127]:
conditions = [
              (df_master_validation['model1'] == 1) & (df_master_validation['model2'] == 1) & (df_master_validation['model3'] >= 0.5),
              (df_master_validation['model1'] == 0) & (df_master_validation['model2'] == 0) & (df_master_validation['model3'] < 0.5)
              ]
choices = ['1','0']

df_master_validation['EnsembleModel'] = np.select(conditions, choices, default='Uncertain')

/tmp/ipython-input-2937165510.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_master_validation['EnsembleModel'] = np.select(conditions, choices, default='Uncertain')


In [128]:
print(len(df_master_validation[df_master_validation['EnsembleModel']=='0']))
print(len(df_master_validation[df_master_validation['EnsembleModel']=='1']))
print(len(df_master_validation[df_master_validation['EnsembleModel']=='Uncertain']))

840
249
441


In [131]:
len(df_master_validation[df_master_validation['EnsembleModel']=='0']) + len(df_master_validation[df_master_validation['EnsembleModel']=='1']) + len(df_master_validation[df_master_validation['EnsembleModel']=='Uncertain'])

1530

In [132]:
len(df_master_validation[(df_master_validation['EnsembleModel']=='1') & (df_master_validation['Churn']==1)])

142

In [133]:
len(df_master_validation[(df_master_validation['EnsembleModel']=='Uncertain') & (df_master_validation['Churn']==1)])

51

In [134]:
len(df_master_validation[df_master_validation['Churn']==1])

209

In [135]:
(142+51) / 209 * 100

92.34449760765551